# **All Sessions Runner**

## **1. Project Setup**

In [15]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\candle-fetcher


## **2. Imports**

In [16]:
from src.pipeline import run_all_sessions_pipeline, print_pipeline_result
from src.mt5_fetcher import MT5FetchConfig, fetch_raw_candles, print_fetch_result

## **3. Settings**

In [17]:
# ---------------------------------------------------------------------------
# All Sessions Runner Settings
# ---------------------------------------------------------------------------

SYMBOL = "US100.cash"
TIMEFRAME = "1m"

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

# Optional filters.
# Leave as None if you do not want to use them.
YEARS = None
MONTHS = None
WEEKS = None
DAYS = None
WEEKDAYS = None

OUTPUT_SUFFIX = None
OUTPUT_NAME = "2025-full-year.csv"


# ---------------------------------------------------------------------------
# MT5 Fetch Settings
# ---------------------------------------------------------------------------

FETCH_FROM_MT5 = True

RANGE_NAME = "all_sessions_range"
RAW_OUTPUT_NAME = None

FETCH_START_DATE = START_DATE
FETCH_END_DATE = END_DATE

CHUNK_DAYS = 31

# For all sessions, UTC is simplest.
DATE_TIMEZONE_NAME = "UTC"

## **4. Fetch Raw Candles From MT5**

In [18]:
input_pattern = "*.csv"

if FETCH_FROM_MT5:
    fetch_config = MT5FetchConfig(
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        range_name=RANGE_NAME,
        output_name=RAW_OUTPUT_NAME,
        start_date=FETCH_START_DATE,
        end_date=FETCH_END_DATE,
        chunk_days=CHUNK_DAYS,
        date_timezone_name=DATE_TIMEZONE_NAME,
    )

    fetch_result = fetch_raw_candles(fetch_config)
    print_fetch_result(fetch_result)

    # Only process the CSV that was just fetched.
    input_pattern = fetch_result["filename"]

else:
    print("Skipping MT5 fetch. Loading existing CSV files from data/raw/.")

Fetching chunk: 2025-01-01 00:00 -> 2025-02-01 00:00 UTC
Fetching chunk: 2025-02-01 00:00 -> 2025-03-04 00:00 UTC
Fetching chunk: 2025-03-04 00:00 -> 2025-04-04 00:00 UTC
Fetching chunk: 2025-04-04 00:00 -> 2025-05-05 00:00 UTC
Fetching chunk: 2025-05-05 00:00 -> 2025-06-05 00:00 UTC
Fetching chunk: 2025-06-05 00:00 -> 2025-07-06 00:00 UTC
Fetching chunk: 2025-07-06 00:00 -> 2025-08-06 00:00 UTC
Fetching chunk: 2025-08-06 00:00 -> 2025-09-06 00:00 UTC
Fetching chunk: 2025-09-06 00:00 -> 2025-10-07 00:00 UTC
Fetching chunk: 2025-10-07 00:00 -> 2025-11-07 00:00 UTC
Fetching chunk: 2025-11-07 00:00 -> 2025-12-08 00:00 UTC
Fetching chunk: 2025-12-08 00:00 -> 2026-01-02 00:00 UTC
MT5 raw candle fetch complete
Symbol: US100.cash
Timeframe: M1
Rows saved: 348,793
First datetime UTC: 2025-01-02 01:05:00+00:00
Last datetime UTC: 2025-12-31 23:49:00+00:00
Filename: us100_cash_M1_raw_all_sessions_range_2025-01-01_to_2026-01-01.csv
Path: C:\candle-fetcher\data\raw\us100_cash_M1_raw_all_sessions_ra

## **5. Run All Sessions Filter**

In [19]:
result = run_all_sessions_pipeline(
    start_date=START_DATE,
    end_date=END_DATE,
    years=YEARS,
    months=MONTHS,
    weeks=WEEKS,
    days=DAYS,
    weekdays=WEEKDAYS,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    input_pattern=input_pattern,
    output_suffix=OUTPUT_SUFFIX,
    output_name=OUTPUT_NAME,
)

print_pipeline_result(result)

Starting candle pipeline
All Sessions: full day / no session filter
Loading: C:\candle-fetcher\data\raw\us100_cash_M1_raw_all_sessions_range_2025-01-01_to_2026-01-01.csv
Raw rows loaded: 348,793
Raw first datetime: 2025-01-02 01:05:00+00:00
Raw last datetime: 2025-12-31 23:49:00+00:00
Rows after date filters: 348,793
Rows after session filter: 348,793
Export complete
Session: all_sessions
Rows saved: 348,793
Filename: 2025_full_year.csv
Path: C:\candle-fetcher\data\processed\all_sessions\2025_full_year.csv
Pipeline result
Session: all_sessions
Description: All Sessions: full day / no session filter
Raw rows: 348,793
Date-filtered rows: 348,793
Final rows: 348,793
First datetime: 2025-01-02 01:05:00+00:00
Last datetime: 2025-12-31 23:49:00+00:00
Saved to: C:\candle-fetcher\data\processed\all_sessions\2025_full_year.csv


## **6. View Data**

In [20]:
df = result["data"]
df.head()

,datetime,open,high,low,close,tick_volume,spread,real_volume,session_name
0,2025-01-02 01:05:00+00:00,21100.15,21104.25,21088.65,21093.65,153,290,0,all_sessions
1,2025-01-02 01:06:00+00:00,21093.15,21093.75,21086.05,21091.35,119,290,0,all_sessions
2,2025-01-02 01:07:00+00:00,21091.65,21092.05,21086.15,21088.25,73,290,0,all_sessions
3,2025-01-02 01:08:00+00:00,21088.85,21097.25,21087.25,21096.05,70,290,0,all_sessions
4,2025-01-02 01:09:00+00:00,21098.05,21099.25,21091.75,21092.15,85,290,0,all_sessions


## **7. Check Size**

In [21]:
df.shape

(348793, 9)